In [1]:
import tensorflow as tf
import keras
import numpy as np
import sys
import matplotlib.pyplot as plt

sys.path.append("../")

In [2]:
DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR= "../models"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/sig_pixel_spacetime.npy"
BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/bg_pixel_spacetime.npy"
SIGNAL_MPPC_FILE = f"{DATA_DIR}/sig_mppc_spacetime.npy"
BACKGROUND_MPPC_FILE = f"{DATA_DIR}/bg_mppc_spacetime.npy"
SIGNAL_ONLY_PIXEL_FILE = f"{DATA_DIR}/sig_only_pixel_spacetime.npy"
SIGNAL_ONLY_MPPC_FILE = f"{DATA_DIR}/sig_only_mppc_spacetime.npy"


bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
bg_mppc_spacetime = np.load(BACKGROUND_MPPC_FILE)
sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
sig_mppc_spacetime = np.load(SIGNAL_MPPC_FILE)


X_pixel = np.concatenate((bg_pixel_spacetime, sig_pixel_spacetime), axis=0)
X_mppc = np.concatenate((bg_mppc_spacetime, sig_mppc_spacetime), axis=0)
y = np.concatenate(
    (np.zeros(bg_pixel_spacetime.shape[0]), np.ones(sig_pixel_spacetime.shape[0])),
    axis=0,
)

# Shuffle the data
indices = np.arange(len(y))
np.random.shuffle(indices)
X_pixel = X_pixel[indices]
X_mppc = X_mppc[indices]
y = y[indices]

input_seq_len = bg_pixel_spacetime.shape[1]
input_dim = bg_pixel_spacetime.shape[2]  # Exclude timestamp

In [3]:
from src.model.components import (
    SelfAttentionStack,
    SelfAttentionBlock,
    CrossAttentionBlock,
    PoolingAttentionBlock,
    MultiHeadAttentionBlock,
    GenerateMask,
    MLP,
)

feature_dim = 8
num_heads = 8
latent_dim = 8
num_seeds = 2
dropout_rate = 0

pixel_input = keras.Input(shape=(input_seq_len, input_dim), name="pixel_input")
mppc_input = keras.Input(shape=(input_seq_len, input_dim), name="mppc_input")
pixel_mask = GenerateMask(name="mask")(pixel_input)
pixel_embedding = MLP(
    num_layers=4,
    output_dim=feature_dim,
    activation="relu",
    name="pixel_embedding",
    dropout_rate=dropout_rate,
)(pixel_input)

mppc_mask = GenerateMask(name="mppc_mask")(mppc_input)
mppc_embedding = MLP(
    num_layers=4,
    output_dim=feature_dim,
    activation="relu",
    name="mppc_embedding",
    dropout_rate=dropout_rate,
)(mppc_input)

comb_seq = keras.layers.Concatenate(name="comb_seq", axis = 1)([pixel_embedding, mppc_embedding])
comb_mask = keras.layers.Concatenate(name="comb_mask", axis = 1)([pixel_mask, mppc_mask])
comb_self_attention = SelfAttentionStack(
    stack_size=3,
    num_heads=num_heads,
    key_dim=feature_dim,
    dropout_rate=dropout_rate,
    name="comb_self_attention",
)(comb_seq, comb_mask)

comb_pooling_attention = PoolingAttentionBlock(
    num_heads=num_heads,
    key_dim=feature_dim,
    num_seeds=num_seeds,
    dropout_rate=dropout_rate,
    name="comb_pooling_attention",
)(comb_self_attention, comb_mask)

flat_pooled = keras.layers.Flatten(name="flat_pooled")(comb_pooling_attention)

comb_latent_dim = MLP(
    num_layers=2,
    output_dim=latent_dim,
    activation="relu",
    name="comb_latent_dim",
    dropout_rate=dropout_rate,
)(flat_pooled)


output = MLP(
    num_layers=3,
    output_dim=1,
    activation="sigmoid",
    name="output",
    dropout_rate=dropout_rate,
)(comb_latent_dim)

model = keras.Model(
    inputs=[pixel_input, mppc_input],
    outputs=output,
    name="mu3e_trigger_model",
)
model.compile(
    optimizer=keras.optimizers.Lion(learning_rate=1e-4),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy()],
)


In [4]:
model.summary()

Model: "mu3e_trigger_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ pixel_input         │ (None, 128, 4)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mppc_input          │ (None, 128, 4)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_embedding     │ (None, 128, 8)    │        137 │ pixel_input[0][0] │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mppc_embedding      │ (None, 128, 8)    │        137 │ mppc_input[0][0]  │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mask (GenerateMask) │ (None, 128, 1)    │          0 │ pixel_input[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mppc_mask           │ (None, 128, 1)    │          0 │ mppc_input[0][0]  │
│ (GenerateMask)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ comb_seq            │ (None, 256, 8)    │          0 │ pixel_embedding[… │
│ (Concatenate)       │                   │            │ mppc_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ comb_mask           │ (None, 256, 1)    │          0 │ mask[0][0],       │
│ (Concatenate)       │                   │            │ mppc_mask[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ comb_self_attention │ (None, 256, 8)    │      7,680 │ comb_seq[0][0],   │
│ (SelfAttentionStac… │                   │            │ comb_mask[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ comb_pooling_atten… │ (None, 2, 8)      │      2,872 │ comb_self_attent… │
│ (PoolingAttentionB… │                   │            │ comb_mask[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flat_pooled         │ (None, 16)        │          0 │ comb_pooling_att… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ comb_latent_dim     │ (None, 8)         │        283 │ flat_pooled[0][0] │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (MLP)        │ (None, 1)         │         49 │ comb_latent_dim[… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 11,158 (43.59 KB)

 Trainable params: 11,158 (43.59 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
from sklearn.model_selection import train_test_split

(
    X_pixel_train,
    X_pixel_test,
    X_mppc_train,
    X_mppc_test,
    y_train,
    y_test,
) = train_test_split(
    X_pixel,
    X_mppc,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)


In [ ]:
model.compile(
    optimizer=keras.optimizers.Lion(learning_rate=1e-4),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy()],
)
model.fit(
    x=[X_pixel_train, X_mppc_train],
    y=y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=512,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True
        )
    ],
    class_weight = {label: 1/np.mean(y_train == label) for label in np.unique(y_train)}
)

Epoch 1/100


/Users/simi/mu3e_trigger/venv/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'self_attention_block' (of type SelfAttentionBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/Users/simi/mu3e_trigger/venv/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'self_attention_block_1' (of type SelfAttentionBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/Users/simi/mu3e_trigger/venv/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'self_attention_block_2' (of type SelfAttentionBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore d

148/282 ━━━━━━━━━━━━━━━━━━━━ 8:38 4s/step - binary_accuracy: 0.4329 - loss: 1.3852

In [ ]:
print(f"Test accuracy: {model.evaluate([X_pixel_test, X_mppc_test], y_test)[1]}")
model.save(f"{MODEL_DIR}/test_class_model.keras")

In [ ]:
model.get_weights()

In [ ]:
model = keras.models.load_model(
    f"{MODEL_DIR}/test_class_model.keras",
    custom_objects={
        "SelfAttentionStack": SelfAttentionStack,
        "SelfAttentionBlock": SelfAttentionBlock,
        "CrossAttentionBlock": CrossAttentionBlock,
        "PoolingAttentionBlock": PoolingAttentionBlock,
        "MultiHeadAttentionBlock": MultiHeadAttentionBlock,
        "GenerateMask": GenerateMask,
        "MLP": MLP,
    },)

print(f"Test accuracy: {model.evaluate([X_pixel_test, X_mppc_test], y_test)[1]}")

In [ ]:
test_seq_length = (X_pixel_test != -1).all(axis=-1).sum(axis=-1) + (X_mppc_test != -1).all(
    axis=-1
).sum(axis=-1)
test_mppc_length = (X_mppc_test != -1).all(axis=-1).sum(axis=-1)
test_pixel_length = (X_pixel_test != -1).all(axis=-1).sum(axis=-1)
train_mppc_length = (X_mppc_train != -1).all(axis=-1).sum(axis=-1)
train_pixel_length = (X_pixel_train != -1).all(axis=-1).sum(axis=-1)

In [ ]:
mppc_lenght_input = keras.Input(shape=(1,), name="mppc_length_input")
pixel_length_input = keras.Input(shape=(1,), name="pixel_length_input")

input = keras.layers.Concatenate(name="input")(
    [
        mppc_lenght_input,
        pixel_length_input,
    ]
)
encoder = MLP(
    num_layers=3,
    output_dim=10,
    name="encoder",
    activation="relu",
)(input)
decoder = MLP(
    num_layers=3,
    output_dim=1,
    name="decoder",
    activation="sigmoid",
)(encoder)
seq_length_mlp = keras.Model(
    inputs=[mppc_lenght_input, pixel_length_input],
    outputs=decoder,
    name="SeqLengthMLP",
)

In [ ]:
seq_length_mlp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy()],
)
seq_length_mlp.summary()

In [ ]:
seq_length_mlp.fit(
    x=[train_mppc_length, train_pixel_length],
    y=y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=128,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True
        )
    ],
    class_weight={label: 1/np.mean(y_train == label) for label in np.unique(y_train) }
)

In [ ]:
test_predictions = model.predict([X_pixel_test, X_mppc_test])
test_seq_length = seq_length_mlp.predict([test_mppc_length, test_pixel_length])

from sklearn.metrics import confusion_matrix, roc_curve, auc

fpr, tpr, thresholds = roc_curve(y_test, test_predictions)
fpr_seq_length, tpr_seq_length, thresholds_seq_length = roc_curve(y_test, test_seq_length)
roc_auc_seq_length = auc(fpr_seq_length, tpr_seq_length)
roc_auc = auc(fpr, tpr)
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color="blue", label="ROC curve (area = {:.2f})".format(roc_auc))
plt.plot(
    fpr_seq_length,
    tpr_seq_length,
    color="green",
    label="MLP trained on number of hits of MPPC and Pixels (area = {:.2f})".format(roc_auc_seq_length),
)
plt.title("Signal only")
plt.grid()
plt.plot([0, 1], [0, 1], color="red", linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.savefig("roc_curve.png")